# Getting started with Atlas Profiler

This notebook profiles a public dataset of NYC open spaces (14,923 parks, cemeteries and other places, with their coordinates) and walks through the profile: what was computed on the sample, the column types and statistics, where the data is (bounding box, k-d tree and density grid), a histogram, and how to save the profile.

In [1]:
import json
import time

import pandas as pd
import polars as pl

from atlas_profiler import process_dataset

## Load the data

The dataset is a Parquet file on the Hugging Face Hub; polars reads `hf://` paths directly. Converting it to pandas needs `pyarrow` (`pip install pyarrow`), which the profiler itself does not need.

In [2]:
df = pl.read_parquet(
    "hf://datasets/oscur/NYC_public_open_spaces/data/train-00000-of-00001.parquet"
).to_pandas()
df.head()

,NAME,STATUS,index_right,PHYSICALID,source,segmentidt,Borough Name,lon,lat,geometry_wkt
0,Bayside-Acacia-Mokom Sholom Cemetery,Unchanged,100766.0,30644.0,open_space_other,NaN,NaN,-73.854919,40.677690,"MULTIPOLYGON (((-73.852549 40.675407, -73.8525..."
1,Cypress Hills-Mt Hope-Maimonides Cemetery,Unchanged,48271.0,15071.0,open_space_other,NaN,NaN,-73.873689,40.693060,"MULTIPOLYGON (((-73.869944 40.69216, -73.86993..."
2,Union Field-Beth El-Machpelah-Hungarian Cemetery,Unchanged,26106.0,168005.0,open_space_other,NaN,NaN,-73.888931,40.693425,"MULTIPOLYGON (((-73.884977 40.694323, -73.8849..."
3,NO NAME,Unchanged,NaN,NaN,open_space_other,NaN,NaN,-73.898839,40.666883,"MULTIPOLYGON (((-73.898797 40.666958, -73.8987..."
4,Holy Cross Cemetery,Unchanged,58373.0,106197.0,open_space_other,NaN,NaN,-73.938267,40.647000,"MULTIPOLYGON (((-73.938569 40.649485, -73.9385..."


## Profile it

`plots=True` adds a histogram per column. The sample size is chosen from the table (about 2 million values, at least 10,000 rows), so this table is profiled whole. The spatial classifier (`geo_classifier=True`, the default) downloads its model files the first time it runs.

In [3]:
start = time.time()
metadata = process_dataset(df, plots=True, indexes=False)
print(f"Profiled {metadata['nb_profiled_rows']:,} of {metadata['nb_rows']:,} rows in {time.time() - start:.1f} s")
metadata["computed_on"]

Profiled 14,923 of 14,923 rows in 4.2 s


{'column_types': 'all_rows',
 'value_ranges': 'all_rows',
 'histograms': 'all_rows',
 'distinct_counts': 'all_rows',
 'numeric_statistics': 'all_rows',
 'spatial_coverage': 'all_rows',
 'temporal_coverage': 'all_rows'}

## Columns

Types are schema.org URLs; `geo_classifier` holds the spatial classifier's label and confidence.

In [4]:
short = lambda uri: uri.rsplit('/', 1)[-1]
pd.DataFrame([
    {
        "column": c["name"],
        "type": short(c["structural_type"]),
        "semantic": ", ".join(short(t) for t in c.get("semantic_types", [])),
        "classifier": (c.get("geo_classifier") or {}).get("label", ""),
        "missing": round(c.get("missing_values_ratio", 0.0), 3),
        "distinct": c.get("num_distinct_values"),
        "mean": c.get("mean"),
        "min": c.get("min"),
        "max": c.get("max"),
    }
    for c in metadata["columns"]
])

,column,type,semantic,classifier,missing,distinct,mean,min,max
0,NAME,Text,Enumeration,,0.580,83.0,NaN,None,None
1,STATUS,Text,Enumeration,,0.029,3.0,NaN,None,None
2,index_right,Integer,identifier,,0.751,2719.0,59363.508748,None,None
3,PHYSICALID,Integer,identifier,,0.751,2718.0,102660.988156,None,None
4,source,Text,Enumeration,,0.000,2.0,NaN,None,None
5,segmentidt,Integer,,,0.971,377.0,149109.437209,None,None
6,Borough Name,Text,AdministrativeArea,borough,0.000,NaN,NaN,None,None
7,lon,Float,longitude,longitude,0.000,NaN,-73.941067,None,None
8,lat,Float,latitude,latitude,0.000,NaN,40.695559,None,None
9,geometry_wkt,GeoShape,,multi-polygon,0.000,NaN,NaN,None,None


## Spatial coverage

For each lat/long pair: a bounding box (0.1–99.9% of each coordinate for a CSV path, robust clusters for a DataFrame), the count of (0, 0) placeholders, a k-d tree whose leaves each hold about the same number of points, and an exact density grid of square cells over the area holding most of the points.

In [5]:
for entry in metadata.get("spatial_coverage", []):
    if entry["type"] != "latlong":
        continue
    print("Columns:", entry["column_names"])
    for rng in entry["ranges"]:
        (min_long, max_lat), (max_long, min_lat) = rng["range"]["coordinates"]
        print(f"  box: lon {min_long:.4f} to {max_long:.4f}, lat {min_lat:.4f} to {max_lat:.4f}")
    print(f"  points: {entry['number']:,}; at (0, 0): {entry.get('number_zero_coordinates', 0):,}")
    leaves = entry["kdtree"]["leaves"]
    print(f"  k-d tree: {len(leaves):,} leaves of about {leaves[0]['number']} points")
    grid = entry["grid"]
    print(f"  grid: {grid['columns']} x {grid['rows']} cells of {1000 * grid['cell_km']:.0f} m, "
          f"{len(grid['cells']):,} non-empty, {grid['outside']:,} points outside")

Columns: ['lat', 'lon']
  box: lon -74.2310 to -74.0240, lat 40.5235 to 40.6402
  box: lon -73.9959 to -73.7605, lat 40.5881 to 40.7319
  box: lon -73.9561 to -73.7465, lat 40.7456 to 40.8922
  points: 14,923; at (0, 0): 0
  k-d tree: 1,024 leaves of about 14 points
  grid: 201 x 200 cells of 339 m, 3,976 non-empty, 0 points outside


The grid can be turned into a table (or an image) directly: each non-empty cell is `[column, row, count]`, with column 0 at the grid's west edge and row 0 at its south edge. Here are its densest cells:

In [6]:
entry = next(e for e in metadata["spatial_coverage"] if e["type"] == "latlong")
grid = entry["grid"]
x0, y0 = grid["bounds"][:2]
dx, dy = grid["cell_degrees"]
cells = pd.DataFrame(grid["cells"], columns=["column", "row", "count"])
cells["lon"] = x0 + (cells["column"] + 0.5) * dx
cells["lat"] = y0 + (cells["row"] + 0.5) * dy
cells.sort_values("count", ascending=False).head()

,column,row,count,lon,lat
349,72,57,120,-74.088197,40.576735
350,73,57,88,-74.084181,40.576735
648,150,64,84,-73.774965,40.598190
320,72,56,74,-74.088197,40.573671
348,71,57,62,-74.092212,40.576735


## A histogram

Numeric histograms bin a robust range and count the values below and above it; integer columns with few values get one bar per value.

In [7]:
lat = next(c for c in metadata["columns"] if c["name"].lower().startswith("lat"))
plot = lat["plot"]
print(f"{lat['name']}: {len(plot['data'])} bins, {plot['below']} values below and {plot['above']} above the range")
pd.DataFrame(plot["data"])

lat: 20 bins, 75 values below and 75 above the range


,count,bin_start,bin_end
0,226,40.511055,40.530686
1,370,40.530686,40.550317
2,402,40.550317,40.569949
3,1314,40.569949,40.589580
4,1392,40.589580,40.609211
5,752,40.609211,40.628843
6,873,40.628843,40.648474
7,834,40.648474,40.668105
8,1570,40.668105,40.687737
9,1095,40.687737,40.707368


## Temporal coverage

This dataset has no date columns, so the list is empty; see the documentation's examples for the fields of a date column.

In [8]:
metadata.get("temporal_coverage", [])

[]

## Save the profile

`default=str` writes the few values JSON has no type for.

In [9]:
with open("nyc_open_spaces.profile.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)